## UC00206: Project 35 - School Enrolment Trends

**Objective:** Support city-wide education planning and infrastructure development using enrolment data.

The initial enrolment data available was only for 2024. I sourced additional data for 2022, 2023, and 2025 from online sources to enable trend analysis. I am analysing Victorian school enrolment trends over these four years and combining them with school location data to understand how student numbers vary by school type, sector, and region.

I collected enrolment datasets from GitHub and the school location dataset via an API. Each dataset had slightly different columns, so I cleaned and standardised the column names, removed inconsistent columns, and ensured data types matched.  

Next, I filtered the enrolment data to keep only schools that were open ("School_Status" = "O") and present in all four years, ensuring consistency for trend analysis. I also handled duplicate school numbers by combining the `Entity_Type` with `School_No` to create unique school IDs for both the enrolment and location datasets.  

Finally, I merged the cleaned enrolment data with the location dataset using these unique IDs. The resulting dataset contains both enrolment and location information for each school, ready for trend analysis and further insigsis.

In [2]:
import pandas as pd
import requests
from io import StringIO

In [3]:
# Import the school locations data set using API
base_url = "https://discover.data.vic.gov.au/api/3/action/datastore_search"

resource_id = "a70cb615-ed28-48b6-bebe-2d0d346f7927"

url = f"{base_url}?resource_id={resource_id}&limit=5000"

response = requests.get(url)

if response.status_code == 200:

    data = response.json()

    school_locations_df = pd.DataFrame(data["result"]["records"])

    print(school_locations_df.head())

else:

    print(f"Request failed with status code {response.status_code}")

   _id Education_Sector Entity_Type School_No                    School_Name  \
0    1         Catholic           2        20                 Parade College   
1    2         Catholic           2        25       Simonds Catholic College   
2    3         Catholic           2        26    St Mary’s College Melbourne   
3    4         Catholic           2        28  St Patrick's College Ballarat   
4    5         Catholic           2        29            St Patrick's School   

  School_Type             Address_Line_1 Address_Line_2    Address_Town  \
0   Secondary           1436 Plenty Road           None        BUNDOORA   
1   Secondary        273 Victoria Street           None  WEST MELBOURNE   
2   Secondary         11 Westbury Street           None   ST KILDA EAST   
3   Secondary          1431 Sturt Street           None        BALLARAT   
4     Primary  119 Drummond Street South           None        BALLARAT   

  Address_State  ...     Postal_Town Postal_State Postal_Postcode  \

In [4]:
# load the enrolment data for 2022 to 2025 from the GitHub repo)
files = {
    "dv335-allschoolsFTEenrolmentsFeb2022.csv": "https://raw.githubusercontent.com/Chameleon-company/MOP-Code/master/datascience/usecases/DEPENDENCIES/UC00206_Project%2035_Source%20Data/dv335-allschoolsFTEenrolmentsFeb2022.csv",
    "dv355-VIC All Schools Enrolments 2023.csv": "https://raw.githubusercontent.com/Chameleon-company/MOP-Code/master/datascience/usecases/DEPENDENCIES/UC00206_Project%2035_Source%20Data/dv355-VIC%20All%20Schools%20Enrolments%202023.csv",
    "dv377_DataVic-AllSchoolsEnrolments-2024.csv": "https://raw.githubusercontent.com/Chameleon-company/MOP-Code/master/datascience/usecases/DEPENDENCIES/UC00206_Project%2035_Source%20Data/dv377_DataVic-AllSchoolsEnrolments-2024.csv",
    "dv403-AllSchoolsEnrolments-2025.csv": "https://raw.githubusercontent.com/Chameleon-company/MOP-Code/master/datascience/usecases/DEPENDENCIES/UC00206_Project%2035_Source%20Data/dv403-AllSchoolsEnrolments-2025.csv",
}

dfs = []
for fname, url in files.items():
    df = pd.read_csv(url, encoding="latin1")
    df.columns = (
        df.columns
        .str.strip()
        .str.replace('"', '', regex=False)
        .str.replace(' ', '_')
    )
    df["School_No"] = df["School_No"].astype(str).str.strip()
    # Assign Census_Year based on filename
    if "2022" in fname:
        df["Census_Year"] = 2022
    elif "2023" in fname:
        df["Census_Year"] = 2023
    elif "2024" in fname:
        df["Census_Year"] = 2024
    elif "2025" in fname:
        df["Census_Year"] = 2025
    dfs.append(df)

enrolments_all_df = pd.concat(dfs, ignore_index=True)
print(enrolments_all_df.shape)
print(enrolments_all_df.head())


(9172, 34)
  Education_Sector  Entity_Type School_No                  School_Name  \
0         Catholic            2       204           St Monica's School   
1         Catholic            2       205           St Joseph's School   
2         Catholic            2       212  St Brendan's Primary School   
3         Catholic            2       219         Sacred Heart College   
4         Catholic            2       227          St Patrick's School   

  School_Type School_Status  Prep_Total  Year_1_Total  Year_2_Total  \
0     Primary             O        47.0          34.0          58.0   
1     Primary             O        52.0          55.0          42.0   
2     Primary             O         1.0           2.0           3.0   
3   Secondary             O         0.0           0.0           0.0   
4     Primary             O        17.0          23.0          21.0   

   Year_3_Total  ...  Year  CENSUS_TYPE  Census_Year  DE_Admin_Region  \
0          55.0  ...  2022            F     

In [5]:
# Find common columns

# Get columns from each DataFrame as a set
columns_sets = [set(df.columns) for df in dfs]

# common columns
common_columns = list(set.intersection(*columns_sets))  # convert set to list
print("Common columns in all 4 files:")
print(common_columns)

Common columns in all 4 files:
['Year_8_Total', 'Secondary_Total', 'Census_Year', 'Grand_Total', 'Secondary_Ungraded_Total', 'Entity_Type', 'Year_1_Total', 'Year_12_Total', 'Year_2_Total', 'CENSUS_TYPE', 'Primary_Ungraded_Total', 'Year_7_Total', 'Year_6_Total', 'School_Status', 'Year_4_Total', 'Primary_Total', 'School_Type', 'Prep_Total', 'Year_5_Total', 'Year_11_Total', 'School_No', 'Year', 'School_Name', 'Year_10_Total', 'Year_3_Total', 'Year_9_Total']


In [6]:
#Filter common columns
enrolments_all_df = pd.concat([df[common_columns] for df in dfs], ignore_index=True)
print(enrolments_all_df.shape)
print(enrolments_all_df.head(5))

(9172, 26)
   Year_8_Total  Secondary_Total  Census_Year  Grand_Total  \
0           0.0              0.0         2022        316.0   
1           0.0              0.0         2022        351.0   
2           0.0              0.0         2022         15.0   
3         239.0           1463.0         2022       1463.0   
4           0.0              0.0         2022        110.0   

   Secondary_Ungraded_Total  Entity_Type  Year_1_Total  Year_12_Total  \
0                       0.0            2          34.0            0.0   
1                       0.0            2          55.0            0.0   
2                       0.0            2           2.0            0.0   
3                       0.0            2           0.0          243.0   
4                       0.0            2          23.0            0.0   

   Year_2_Total CENSUS_TYPE  ...  School_Type  Prep_Total  Year_5_Total  \
0          58.0           F  ...      Primary        47.0          46.0   
1          42.0           F

In [7]:
# Check unique School_No per Census_Year
for year in enrolments_all_df["Census_Year"].unique():
    df_year = enrolments_all_df[enrolments_all_df["Census_Year"] == year]
    total_schools = len(df_year)
    unique_schools = df_year["School_No"].nunique()
    print(f"Year {year}: Total rows = {total_schools}, Unique School_No = {unique_schools}")

Year 2022: Total rows = 2286, Unique School_No = 2145
Year 2023: Total rows = 2290, Unique School_No = 2149
Year 2024: Total rows = 2295, Unique School_No = 2155
Year 2025: Total rows = 2301, Unique School_No = 2162


In [8]:
# Check one example of duplicate School_No per year
for year in enrolments_all_df["Census_Year"].unique():
    df_year = enrolments_all_df[enrolments_all_df["Census_Year"] == year]
    
    # Find duplicated School_No
    duplicated_schools = df_year[df_year.duplicated(subset=["School_No"], keep=False)]
    
    if not duplicated_schools.empty:
        # Get the first duplicate School_No
        first_dup_school_no = duplicated_schools["School_No"].iloc[0]
        
        print(f"\nYear {year} - Example duplicate School_No: {first_dup_school_no}")
        print(duplicated_schools[duplicated_schools["School_No"] == first_dup_school_no])


Year 2022 - Example duplicate School_No: 250
     Year_8_Total  Secondary_Total  Census_Year  Grand_Total  \
6           206.0           1209.0         2022       1209.0   
492           0.0              0.0         2022        332.0   

     Secondary_Ungraded_Total  Entity_Type  Year_1_Total  Year_12_Total  \
6                         0.0            2           0.0          194.0   
492                       0.0            1          60.0            0.0   

     Year_2_Total CENSUS_TYPE  ...  School_Type  Prep_Total  Year_5_Total  \
6             0.0           F  ...    Secondary         0.0           0.0   
492          61.0           F  ...      Primary        63.0          26.0   

    Year_11_Total  School_No  Year                School_Name  Year_10_Total  \
6           207.0        250  2022    Star of the Sea College          201.0   
492           0.0        250  2022  Flemington Primary School            0.0   

     Year_3_Total  Year_9_Total  
6             0.0         19

### Duplicate school numbers have differnt Entity Types and Sector, I'll need to create a unique school ID to avoid dupicates 

In [10]:
# Check unique Entity_Type values per year
entity_types_per_year = (
    enrolments_all_df.groupby("Census_Year")["Entity_Type"]
    .unique()
    .reset_index()
    .rename(columns={"Entity_Type": "Unique_Entity_Types"})
)

print(entity_types_per_year)

   Census_Year Unique_Entity_Types
0         2022              [2, 1]
1         2023              [2, 1]
2         2024              [2, 1]
3         2025              [2, 1]


In [11]:
# Create a unique school ID by prefixing based on Entity_Type
def create_unique_id(row):
    if row["Entity_Type"] == 1:
        return 100_000 + int(row["School_No"])
    elif row["Entity_Type"] == 2:
        return 200_000 + int(row["School_No"])
    else:
        return int(row["School_No"])  # fallback, just in case

enrolments_all_df["Unique_School_ID"] = enrolments_all_df.apply(create_unique_id, axis=1)

# Check the first few rows
print(enrolments_all_df[["School_No", "Entity_Type", "Unique_School_ID"]].head(10))

  School_No  Entity_Type  Unique_School_ID
0       204            2            200204
1       205            2            200205
2       212            2            200212
3       219            2            200219
4       227            2            200227
5       236            2            200236
6       250            2            200250
7       251            2            200251
8       252            2            200252
9       262            2            200262


In [12]:
# Count of schools by School_Status and Census_Year
status_counts = enrolments_all_df.groupby(["Census_Year", "School_Status"])["Unique_School_ID"].nunique().reset_index()

# Rename for clarity
status_counts.rename(columns={"Unique_School_ID": "School_Count"}, inplace=True)

print(status_counts)

   Census_Year School_Status  School_Count
0         2022             C             1
1         2022             O          2284
2         2022             U             1
3         2023             C             1
4         2023             O          2286
5         2023             U             3
6         2024             O          2294
7         2024             U             1
8         2025             C             4
9         2025             O          2297


In [13]:
# Keep only open schools
open_schools_df = enrolments_all_df[enrolments_all_df["School_Status"] == "O"]

# Count the number of years each school appears in the dataset
school_year_counts = open_schools_df.groupby("Unique_School_ID")["Census_Year"].nunique().reset_index()
school_year_counts.rename(columns={"Census_Year": "Year_Count"}, inplace=True)

# Keep only schools that appear in all 4 years
schools_all_years = school_year_counts[school_year_counts["Year_Count"] == 4]["Unique_School_ID"]

# Filter the open_schools_df
open_schools_all_years_df = open_schools_df[open_schools_df["Unique_School_ID"].isin(schools_all_years)]

# Check result
print(open_schools_all_years_df.groupby("Census_Year")["Unique_School_ID"].nunique())

Census_Year
2022    2246
2023    2246
2024    2246
2025    2246
Name: Unique_School_ID, dtype: int64


In [14]:
print(open_schools_all_years_df.shape)
open_schools_all_years_df.head()

(8984, 27)


,Year_8_Total,Secondary_Total,Census_Year,Grand_Total,Secondary_Ungraded_Total,Entity_Type,Year_1_Total,Year_12_Total,Year_2_Total,CENSUS_TYPE,...,Prep_Total,Year_5_Total,Year_11_Total,School_No,Year,School_Name,Year_10_Total,Year_3_Total,Year_9_Total,Unique_School_ID
0,0.0,0.0,2022,316.0,0.0,2,34.0,0.0,58.0,F,...,47.0,46.0,0.0,204,2022,St Monica's School,0.0,55.0,0.0,200204
1,0.0,0.0,2022,351.0,0.0,2,55.0,0.0,42.0,F,...,52.0,46.0,0.0,205,2022,St Joseph's School,0.0,66.0,0.0,200205
3,239.0,1463.0,2022,1463.0,0.0,2,0.0,243.0,0.0,F,...,0.0,0.0,249.0,219,2022,Sacred Heart College,235.0,0.0,260.0,200219
4,0.0,0.0,2022,110.0,0.0,2,23.0,0.0,21.0,F,...,17.0,10.0,0.0,227,2022,St Patrick's School,0.0,11.0,0.0,200227
5,145.0,800.0,2022,800.0,0.0,2,0.0,71.0,0.0,F,...,0.0,0.0,109.0,236,2022,St Joseph's College Mildura,151.0,0.0,162.0,200236


In [15]:
# Check unique School_No per Census_Year
for year in open_schools_all_years_df["Census_Year"].unique():
    df_year = open_schools_all_years_df[open_schools_all_years_df["Census_Year"] == year]
    total_schools = len(df_year)
    unique_schools = df_year["Unique_School_ID"].nunique()
    print(f"Year {year}: Total rows = {total_schools}, Unique School_No = {unique_schools}")

Year 2022: Total rows = 2246, Unique School_No = 2246
Year 2023: Total rows = 2246, Unique School_No = 2246
Year 2024: Total rows = 2246, Unique School_No = 2246
Year 2025: Total rows = 2246, Unique School_No = 2246


In [16]:
## lets create a similar unique ID for the location dataset

# Make a copy to avoid SettingWithCopyWarning
school_locations_df = school_locations_df.copy()

# Ensure Entity_Type_loc is integer
school_locations_df["Entity_Type"] = school_locations_df["Entity_Type"].astype(int)

# Create unique school ID
def create_unique_id_loc(row):
    if row["Entity_Type"] == 1:
        return 100_000 + int(row["School_No"])
    elif row["Entity_Type"] == 2:
        return 200_000 + int(row["School_No"])
    else:
        return int(row["School_No"])  # fallback

school_locations_df["Unique_School_ID"] = school_locations_df.apply(create_unique_id_loc, axis=1)

# Check
print(school_locations_df.shape)
print(school_locations_df[["School_No", "Entity_Type", "Unique_School_ID"]].head(10))

(2294, 24)
  School_No  Entity_Type  Unique_School_ID
0        20            2            200020
1        25            2            200025
2        26            2            200026
3        28            2            200028
4        29            2            200029
5        30            2            200030
6        33            2            200033
7        35            2            200035
8        60            2            200060
9        77            2            200077


In [17]:
# Ensure School_No is string and stripped
school_locations_df["Unique_School_ID"] = school_locations_df["Unique_School_ID"].astype(str).str.strip()

# Make a copy to avoid SettingWithCopyWarning
open_schools_all_years_df = open_schools_all_years_df.copy()
open_schools_all_years_df["Unique_School_ID"] = open_schools_all_years_df["Unique_School_ID"].astype(str).str.strip()

# Merge enrolment data with location data
# Use inner join so only schools present in both datasets are kept
combined_df = open_schools_all_years_df.merge(
    school_locations_df,
    on="Unique_School_ID",
    how="inner",  # inner ensures only matching schools are kept
    suffixes=("_enrol", "_loc")
)

print("Final dataset shape:", combined_df.shape)
print("Columns available for analysis:", combined_df.columns.tolist())

Final dataset shape: (8984, 50)
Columns available for analysis: ['Year_8_Total', 'Secondary_Total', 'Census_Year', 'Grand_Total', 'Secondary_Ungraded_Total', 'Entity_Type_enrol', 'Year_1_Total', 'Year_12_Total', 'Year_2_Total', 'CENSUS_TYPE', 'Primary_Ungraded_Total', 'Year_7_Total', 'Year_6_Total', 'School_Status', 'Year_4_Total', 'Primary_Total', 'School_Type_enrol', 'Prep_Total', 'Year_5_Total', 'Year_11_Total', 'School_No_enrol', 'Year', 'School_Name_enrol', 'Year_10_Total', 'Year_3_Total', 'Year_9_Total', 'Unique_School_ID', '_id', 'Education_Sector', 'Entity_Type_loc', 'School_No_loc', 'School_Name_loc', 'School_Type_loc', 'Address_Line_1', 'Address_Line_2', 'Address_Town', 'Address_State', 'Address_Postcode', 'Postal_Address_Line_1', 'Postal_Address_Line_2', 'Postal_Town', 'Postal_State', 'Postal_Postcode', 'Full_Phone_No', 'DE_Admin_Region', 'DE_Admin_AREA', 'LGA_ID', 'LGA_Name', 'X', 'Y']
